| Cell # | 类型 | 内容 |
|--------|------|------|
| 1 | 📝 Markdown | `--- 260723 ---` |
| 2 | 📝 Markdown | reviewer 意见 + 你的思考 |
| 3 | 💻 Code | imports |
| 4 | 💻 Code | `MultiTrackPredictor` 类定义 |
| 5 | 💻 Code | Args + 路径配置 |
| 6 | 💻 Code | 基因长度分布统计 |
| 7 | 💻 Code | 随机种子 + 初始化 predictor |
| **8** | **💻 Code** | **[Cell A] 机器状态 + 加载窗口 + 参数** |
| **9** | **📝 Markdown** | **Cell B 说明** |
| **10** | **💻 Code** | **[Cell B] benchmark_at_length 函数** |
| **11** | **📝 Markdown** | **Cell C 说明** |
| **12** | **💻 Code** | **[Cell C] FASTA 加载** |
| **13** | **📝 Markdown** | **Cell D 说明** |
| **14** | **💻 Code** | **[Cell D] 执行 Benchmark** |
| **15** | **📝 Markdown** | **Cell E 说明** |
| **16** | **💻 Code** | **[Cell E] 结果展示 + 可视化 + 保存** |
| 17 | 💻 Code | 参考/突变序列预测（原始逻辑） |

每个代码 cell 前面都有对应的 markdown 说明，可以按顺序逐段调试。运行顺序：**Cell A → B → C → D → E**，每步都可以检查输出是否正确再继续。

已进行更改。

reviewer: The suggested revision is to report inference time (single sequence, batch) and memory requirements, test and report consumer GPU (RTX 4090/5090) runnability including maximum context length and inference speed, report FLOPs and provide per-FLOP comparison with competing models.
思考：
  - RNA预测一般的场景
    - 统计基因长度的分布
    - 基因区的预测和基因区padding预测？
  - 设置要展示的信息
    - 时间和资源


In [ ]:
import os
import sys
import json
import time
import argparse
import random
import glob
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from transformers import AutoTokenizer, AutoConfig
from safetensors.torch import load_file
import pyfaidx
import dotenv
from pathlib import Path
import re

# 导入自定义模块（需确保 src 目录在 Python 路径中）
from src.dataset import MultiTrackDataset, load_fasta_sequence
from src.viewer import DatasetViewer, ResultsViewer
from src.model import GenOmics, load_finetuned_model

# 设置中文字体（根据系统环境调整）
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
class MultiTrackPredictor:
    def __init__(self, fasta_path: str, base_model_path: str, sft_ckpt_path: str,
                 tokenizer_path: str, use_flash_attn: bool, index_stat_path: str,
                 **kwargs):
        self.fasta = pyfaidx.Fasta(fasta_path)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
        self.model = load_finetuned_model(
            model_class = GenOmics,
            model_path = base_model_path,
            ckpt_path = sft_ckpt_path,
            use_flash_attn = use_flash_attn,
            device = "cuda:0",
            model_init_kwargs={"index_stat:" json.load(open(index_stat_path, "r")),
                               **kwargs}
        )
        print("✔️ Model loaded successfully")
        print(self.model)
        self.model.eval()

    def predict(self, chrom: str, start: int, end: int, biosample_names: list = None) -> dict:
            predict_sequence = load_fasta_sequence(self.fasta, chrom, start, end)
            inputs = self.tokenizer(
                predict_sequence,
                return_tensors="pt",
                padding=False,
                truncation=True,
                max_length=32768,
                add_special_tokens=False
            ).to("cuda")
            with torch.no_grad():
                start_time = time.time()
                result = self.model.predict(inputs['input_ids'],
                                            biosample_names=biosample_names)
                time_taken = time.time() - start_time
                torch.cuda.empty_cache()
            print(f"Inference time: {time_taken:.2f} s.")
            return {
                'sequence': predict_sequence,
                'position': (chrom, start, end),
                'values': result
            }
    def predict2(self, chrom: str, start: int, end: int, seq: str, biosample_names: list = None) -> dict:
        sequence = pyfaidx.Fasta(seq)
        predict_sequence = str(sequence[0][:])
        inputs = self.tokenizer(
            predict_sequence,
            return_tensors="pt",
            padding=False,
            truncation=True,
            max_length=32768,
            add_special_tokens=False
        ).to("cuda")
        with torch.no_grad():
            start_time = time.time()
            result = self.model.predict(input['input_ids'],
                                        biosample_names = biosample_names)
            time_taken = time.time() - start_time
            torch.cuda.empty_cache()
        print(f"Inference time: {time_taken:.2f} s.")
        return {
            'sequence': predict_sequence,
            'position': (chrom, start, end),
            'values': result
        }


In [ ]:
class Args:
    pass

args = Args()
args.mut_fasta = '/mnt/rice/default/Workspace/Rice-Genome/application/RNAseq/mutant_predict/riceNavi_output/mutant_fastas/example_214.alt.fa'
args.output_dir = '/mnt/rice/default/Workspace/yangdong/ai4research/ft-rice/response/inference'
args.biosample_names = "total_RNA"

# 固定配置（请根据实际情况修改）
base_model_dir = "/mnt/rice/default/Workspace/xuxiaolong/rice_1B_stage2_8k_hf"
sft_ckpt_path = "/mnt/rice/default//mnt/rWorkspace/yangdong/gene_expression_prediction/outputs/train/202607161357/checkpoint-2720/model.safetensors"
fasta_path = "/mnt/rice/default/Workspace/Rice-Genome/RNAseq/riceRNAseqData/18k/ref/P7-new.fasta"
ANNOTATION_PATH = "/mnt/rice/default/Workspace/xuxiaolong/RNAprediction/ref/P7.IRGSP.liftoff.gff3"
index_stat_path = "/mnt/rice/default/Workspace/yangdong/gene_expression_prediction/data2/CSQ_4_12/indices/test_CSQ_P7_multitrack/index_stat.json"

In [ ]:
# ============================================================
# 统计基因长度分布（从 GFF 注释文件）
# ============================================================

def parse_gff_attr(attr_text: str) -> dict:
    """解析 GFF3 属性列 (key=value;key=value)"""
    attrs = {}
    for item in attr_text.strip().strip(";").split(";"):
        item = item.strip()
        if not item or "=" not in item:
            continue
        key, value = item.split("=", 1)
        from urllib.parse import unquote
        attrs[key.strip()] = unquote(value.strip())
    return attrs

def load_gene_lengths(gff_path: str) -> pd.DataFrame:
    """从 GFF 文件中提取所有 gene 的 ID、位置和长度"""
    records = []
    with open(gff_path, "r") as f:
        for line in f:
            if not line.strip() or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9:
                continue
            chrom, _, ftype, start, end, _, strand, _, attr_text = parts[:9]
            if ftype != "gene":
                continue
            attrs = parse_gff_attr(attr_text)
            gene_id = (attrs.get("ID") or attrs.get("Name") or 
                       attrs.get("gene_id") or "unknown")
            start_pos, end_pos = int(start), int(end)
            length = end_pos - start_pos + 1  # GFF 是 1-based 闭区间
            records.append({
                "gene_id": gene_id,
                "chrom": chrom,
                "start": start_pos,
                "end": end_pos,
                "strand": strand,
                "length": length,
            })
    df = pd.DataFrame(records)
    return df

print(f"📂 正在读取 GFF 文件: {ANNOTATION_PATH}")
gene_df = load_gene_lengths(ANNOTATION_PATH)
print(f"✅ 共加载 {len(gene_df):,} 个基因\n")

# ---- 基础统计 ----
stats = gene_df["length"].describe(percentiles=[0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95])
print("📊 基因长度分布统计 (bp):")
print(f"  数量:      {len(gene_df):,}")
print(f"  均值:      {stats['mean']:,.0f}")
print(f"  标准差:    {stats['std']:,.0f}")
print(f"  最小值:    {stats['min']:,.0f}")
print(f"  5% 分位:   {stats['5%']:,.0f}")
print(f"  10% 分位:  {stats['10%']:,.0f}")
print(f"  25% 分位:  {stats['25%']:,.0f}")
print(f"  中位数:    {stats['50%']:,.0f}")
print(f"  75% 分位:  {stats['75%']:,.0f}")
print(f"  90% 分位:  {stats['90%']:,.0f}")
print(f"  95% 分位:  {stats['95%']:,.0f}")
print(f"  最大值:    {stats['max']:,.0f}")

# ---- 模型窗口覆盖分析 ----
for ctx_len in [4096, 8192, 16384, 32768]:
    covered = (gene_df["length"] <= ctx_len).sum()
    pct = covered / len(gene_df) * 100
    print(f"\n  ≤ {ctx_len:>6} bp: {covered:>6,} 个基因 ({pct:.1f}%)")

# ---- 绘制分布图 ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 左图: 完整分布
ax = axes[0]
counts, bins, patches = ax.hist(gene_df["length"], bins=100, 
                                 color="steelblue", edgecolor="white", alpha=0.8)
model_lengths = [4096, 8192, 16384, 32768]
colors = ["#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]
labels = ["4kb", "8kb", "16kb", "32kb (max)"]
for l, c, lb in zip(model_lengths, colors, labels):
    ax.axvline(l, color=c, linestyle="--", linewidth=1.5, label=f"Model ctx: {lb}")
ax.set_xlabel("Gene length (bp)")
ax.set_ylabel("Number of genes")
ax.set_title("Gene length distribution (full range)")
ax.legend(fontsize=9)
ax.set_yscale("log")
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"{int(x):,}"))

# 右图: 截断到 50kb 以内（聚焦短基因）
ax = axes[1]
mask = gene_df["length"] <= 50000
ax.hist(gene_df.loc[mask, "length"], bins=80,
        color="steelblue", edgecolor="white", alpha=0.8)
for l, c, lb in zip(model_lengths, colors, labels):
    ax.axvline(l, color=c, linestyle="--", linewidth=1.5)
ax.set_xlabel("Gene length (bp)")
ax.set_ylabel("Number of genes")
ax.set_title(f"Gene length distribution (≤50kb, {mask.sum():,} genes)")

plt.tight_layout()
# plt.savefig(os.path.join(args.output_dir, "gene_length_distribution.png"), 
#             dpi=150, bbox_inches="tight")
plt.show()
# print(f"\n💾 图片已保存至: {os.path.join(args.output_dir, 'gene_length_distribution.png')}")

In [ ]:
# 设置随机种子
seed = 42
random.seed(seed)
np.random.seed(seed)
os.environ["PYTHONHASHSEED"] = str(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)
print(f"✅ Random seed set to {seed}")

# 初始化预测器
predictor = MultiTrackPredictor(
    fasta_path=fasta_path,
    base_model_path=base_model_dir,
    sft_ckpt_path=sft_ckpt_path,
    tokenizer_path=base_model_dir,
    use_flash_attn=True,
    index_stat_path=index_stat_path,
    proj_dim=1024,
    num_downsamples=4,
    bottleneck_dim=1536
)

In [ ]:
# ============================================================
# [Cell A] 机器状态 + 加载测试窗口 + Benchmark 参数
# ============================================================

import psutil

# ---- 1. 机器状态记录 ----
print("=" * 60)
print("🔧 机器状态")
print("=" * 60)
print(f"CPU: {psutil.cpu_count()} cores")
mem = psutil.virtual_memory()
print(f"RAM: {mem.total / 1e9:.1f} GB total, {mem.available / 1e9:.1f} GB available")

print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
    print(f"Compute Capability: {props.major}.{props.minor}")
    print(f"Multi-processors: {props.multi_processor_count}")
print(f"PyTorch version: {torch.__version__}")
print(f"Flash Attention: {'enabled' if predictor.model.base.config._attn_implementation == 'flash_attention_2' else 'disabled'}")

# ---- 2. 加载 P7 Chr01 测试窗口 ----
TEST_CSV = "/mnt/rice/default/Workspace/yangdong/gene_expression_prediction/data2/CSQ_4_12/indices/test_CSQ_P7_multitrack/sequence_split_train.csv"
N_SAMPLE_WINDOWS = 50  # 取前 N 个窗口做 benchmark（避免跑全部 2673 个）

print(f"\n{'='*60}")
print("📂 加载测试窗口")
print(f"{'='*60}")
df_windows = pd.read_csv(TEST_CSV)
print(f"总窗口数: {len(df_windows):,}")

# 筛选 Chr01
chrom_p7 = df_windows["chromosome"].unique()[0]  # 从 CSV 动态获取染色体名
df_chr01 = df_windows[df_windows["chromosome"] == chrom_p7].reset_index(drop=True)
print(f"Chr01 ({chrom_p7}): {len(df_chr01):,} 个窗口")

# 取前 N_SAMPLE_WINDOWS 个用于 benchmark
df_bench = df_chr01.head(N_SAMPLE_WINDOWS).copy()
print(f"Benchmark 采样: {len(df_bench)} 个窗口")

# ---- 3. Benchmark 参数 ----
TARGET_LENGTHS = [2048, 4096, 8192, 16384, 32768]
BATCH_SIZES = [1, 2, 4, 8]
N_WARMUP = 3
N_TRIALS = 5

### Cell B — Benchmark 函数定义

定义核心 benchmark 函数 `benchmark_at_length`：
- 将窗口序列截断到目标长度
- 按 batch_size 分批推理
- 自动 warmup + 多次 trial 取均值
- 记录峰值显存和吞吐量

In [ ]:
# ---- [Cell B] Benchmark 函数 ----
def benchmark_at_length(predictor, seq_list, target_len, batch_size,
                        n_warmup=3, n_trials=5):
    """
    对一组序列截断到 target_len，按 batch_size 分批推理，返回耗时和显存。
    """
    # 截断序列到目标长度
    seqs_trunc = [s[:target_len] for s in seq_list]

    # tokenize 所有序列
    all_input_ids = []
    for s in seqs_trunc:
        tokens = predictor.tokenizer(
            s, return_tensors="pt", padding=False,
            truncation=True, max_length=target_len, add_special_tokens=False
        ).to("cuda")
        all_input_ids.append(tokens['input_ids'])

    # 分批
    n_seqs = len(all_input_ids)
    n_batches = (n_seqs + batch_size - 1) // batch_size

    # ---- warmup ----
    for _ in range(n_warmup):
        for b in range(n_batches):
            batch = all_input_ids[b * batch_size : (b + 1) * batch_size]
            batch_tensor = torch.cat(batch, dim=0)
            with torch.no_grad():
                _ = predictor.model.predict(batch_tensor,
                                            biosample_names=[args.biosample_names])
        torch.cuda.synchronize()
        torch.cuda.empty_cache()
        if hasattr(torch.cuda, 'reset_peak_memory_stats'):
            torch.cuda.reset_peak_memory_stats()

    # ---- 正式计时 ----
    times = []
    for trial in range(n_trials):
        torch.cuda.synchronize()
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        for b in range(n_batches):
            batch = all_input_ids[b * batch_size : (b + 1) * batch_size]
            batch_tensor = torch.cat(batch, dim=0)
            with torch.no_grad():
                _ = predictor.model.predict(batch_tensor,
                                            biosample_names=[args.biosample_names])
        end_event.record()
        torch.cuda.synchronize()
        total_ms = start_event.elapsed_time(end_event)
        times.append(total_ms)
        torch.cuda.empty_cache()

    peak_mem = torch.cuda.max_memory_allocated() / 1e9  # GB

    mean_ms = np.mean(times)
    std_ms = np.std(times)
    throughput_seq_s = n_seqs / (mean_ms / 1000)
    throughput_bp_s = n_seqs * target_len / (mean_ms / 1000)

    return {
        "target_length": target_len,
        "batch_size": batch_size,
        "n_windows": n_seqs,
        "n_batches": n_batches,
        "time_mean_ms": round(mean_ms, 1),
        "time_std_ms": round(std_ms, 1),
        "time_per_seq_ms": round(mean_ms / n_seqs, 2),
        "time_per_batch_ms": round(mean_ms / n_batches, 1),
        "peak_vram_gb": round(peak_mem, 2),
        "throughput_seq_s": round(throughput_seq_s, 1),
        "throughput_bp_s": round(throughput_bp_s),
    }

### Cell C — 加载 FASTA 序列

从 P7 参考基因组中读取 Chr01 前 50 个窗口的 DNA 序列，一次性缓存到内存，避免 benchmark 过程中反复 IO 影响计时。

In [ ]:
# ---- [Cell C] 从 FASTA 加载序列 ----
print("⏳ 正在从 FASTA 加载序列...")
seqs_p7_chr01 = []
for i in range(len(df_bench)):
    row = df_bench.iloc[i]
    chrom = row["chromosome"]
    start = int(row["start"])
    end = int(row["end"])
    seq = load_fasta_sequence(predictor.fasta, chrom, start, end)
    seqs_p7_chr01.append(seq)
print(f"✅ FASTA 序列加载完成，共 {len(seqs_p7_chr01)} 条")

### Cell D — 执行 Benchmark

遍历 5 种序列长度 × 4 种 batch size 共 20 组配置，每组 3 次 warmup + 5 次正式 trial。

In [ ]:
# ---- [Cell D] 执行 Benchmark ----
print(f"\n{'='*60}")
print("🚀 开始 Inference Benchmark")
print(f"  序列长度: {TARGET_LENGTHS}")
print(f"  Batch sizes: {BATCH_SIZES}")
print(f"  每个配置: {N_WARMUP} warmup + {N_TRIALS} trials × {N_SAMPLE_WINDOWS} windows")
print(f"{'='*60}\n")

all_results = []
total_configs = len(TARGET_LENGTHS) * len(BATCH_SIZES)
cfg_idx = 0
for L in TARGET_LENGTHS:
    for B in BATCH_SIZES:
        cfg_idx += 1
        print(f"[{cfg_idx}/{total_configs}]  Length={L:,}  Batch={B}  ...", end=" ", flush=True)
        result = benchmark_at_length(
            predictor, seqs_p7_chr01, L, B,
            n_warmup=N_WARMUP, n_trials=N_TRIALS
        )
        all_results.append(result)
        print(f"✓  {result['time_mean_ms']:.0f}ms  VRAM={result['peak_vram_gb']:.2f}GB  "
              f"吞吐={result['throughput_seq_s']:.0f} seq/s")

df_results = pd.DataFrame(all_results)
df_results["throughput_per_seq_bp_s"] = (df_results["throughput_bp_s"] / 
                                          df_results["batch_size"]).astype(int)

### Cell E — 结果展示与保存

汇总结果表格、三面板可视化（推理时间 / 显存 / 吞吐量热图），并保存 CSV 和 PNG。

In [ ]:
# ---- [Cell E] 结果展示 ----
print(f"\n{'='*60}")
print("📊 Benchmark 汇总结果")
print(f"{'='*60}")
print(df_results.to_string(index=False))

# ---- 可视化 ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = plt.cm.tab10

# 左: 推理时间 vs 序列长度
ax = axes[0]
for bi, B in enumerate(BATCH_SIZES):
    sub = df_results[df_results["batch_size"] == B]
    ax.plot(sub["target_length"], sub["time_mean_ms"], 
            marker="o", label=f"Batch={B}", color=palette(bi), linewidth=1.5)
ax.set_xlabel("Sequence length (bp)")
ax.set_ylabel("Inference time (ms)")
ax.set_title("Inference time vs Length")
ax.legend(fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(TARGET_LENGTHS)
ax.set_xticklabels([f"{l//1000}k" for l in TARGET_LENGTHS])

# 中: Peak VRAM vs 序列长度
ax = axes[1]
for bi, B in enumerate(BATCH_SIZES):
    sub = df_results[df_results["batch_size"] == B]
    ax.plot(sub["target_length"], sub["peak_vram_gb"], 
            marker="s", label=f"Batch={B}", color=palette(bi), linewidth=1.5)
ax.set_xlabel("Sequence length (bp)")
ax.set_ylabel("Peak VRAM (GB)")
ax.set_title("Peak VRAM vs Length")
ax.legend(fontsize=8)
ax.set_xscale("log", base=2)
ax.set_xticks(TARGET_LENGTHS)
ax.set_xticklabels([f"{l//1000}k" for l in TARGET_LENGTHS])
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    ax.axhline(props.total_memory / 1e9, color="red", linestyle="--", 
               label=f"VRAM limit ({props.total_memory/1e9:.0f}GB)")

# 右: 吞吐量热图 (bp/s)
ax = axes[2]
heat_data = df_results.pivot_table(
    index="target_length", columns="batch_size", 
    values="throughput_bp_s", aggfunc="first"
)
im = ax.imshow(heat_data.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(len(BATCH_SIZES)))
ax.set_xticklabels([f"B={b}" for b in BATCH_SIZES])
ax.set_yticks(range(len(TARGET_LENGTHS)))
ax.set_yticklabels([f"{l//1000}k" for l in TARGET_LENGTHS])
ax.set_title("Throughput (bp/s)")
for i in range(len(TARGET_LENGTHS)):
    for j in range(len(BATCH_SIZES)):
        val = heat_data.values[i, j]
        ax.text(j, i, f"{val:,.0f}", ha="center", va="center",
                fontsize=8, color="black" if val < heat_data.values.max()*0.7 else "white")
plt.colorbar(im, ax=ax, fraction=0.046)

plt.tight_layout()
plt.savefig(os.path.join(args.output_dir, "inference_benchmark.png"), 
            dpi=150, bbox_inches="tight")
plt.show()
print(f"\n💾 Benchmark 图已保存至: {os.path.join(args.output_dir, 'inference_benchmark.png')}")

# ---- 保存 CSV ----
csv_path = os.path.join(args.output_dir, "inference_benchmark_results.csv")
df_results.to_csv(csv_path, index=False)
print(f"💾 Benchmark 结果 CSV: {csv_path}")

In [ ]:
# 预测参考序列
print("\n--- Predicting reference sequence ---")
ref_predict = predictor.predict(chrom=chrom, start=window_start, end=window_end,
                                biosample_names=args.biosample_names)

# 预测突变序列
print("\n--- Predicting mutant sequence ---")
mut_predict = predictor.predict2(chrom=chrom, start=window_start, end=window_end,
                                    seq=args.mut_fasta, biosample_names=args.biosample_names)